# AIDA 1141 Project Exercise 05 – Simple Routing Baseline

**Question:** How well do simple, transparent rules identify early-warning assessments that are at risk of missing the start-date deadline, compared with a majority baseline?

This notebook uses the original synthetic files from Exercise 02 and the label defined in Exercise 04. It creates the fixed employee-level split used by Exercises 05–10, chooses rule thresholds on training data, and compares baselines on validation data. Test data is assigned to the split and summarized here, but is not used to choose thresholds or compare baseline performance; it is held back for one final evaluation after the model and review threshold are fixed in Exercise 09.

All rates describe rules applied to simulated outcomes; they are not evidence about real employees or HR performance.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

EX02 = Path("../02-synthetic-request-records")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
SEED = 2026

read = lambda name: pd.read_csv(EX02 / "data" / name, keep_default_na=False, na_values=[""])
requests = read("requests.csv")
assessments = read("assessments.csv")
dictionary = pd.read_csv(EX02 / "data-dictionary.csv", keep_default_na=False)

# Request fields available at assessment time (role model_input in the data dictionary)
request_inputs = dictionary.query("file == 'requests.csv' and role == 'model_input'")["field"].tolist()
data = assessments.merge(requests[["request_id"] + request_inputs], on="request_id", how="left", validate="many_to_one")
data["early_warning"] = ~data["past_deadline_at_assessment"]
print("Request inputs added:", request_inputs)

Request inputs added: ['task_type', 'responsible_team', 'hiring_department', 'employee_type', 'intake_channel', 'info_issue_at_intake']


## A. Data checks

The counts below should match the Exercise 02 data card and the Exercise 04 label guide. Early-warning assessments occur before the 08:00 start-date deadline and are the group the 90% recall target applies to; the 63 overdue assessments are reported separately.

In [2]:
setup_checks = pd.Series({
    "2,579 assessments": len(data) == 2579,
    "2,516 early-warning assessments": data["early_warning"].sum() == 2516,
    "408 early-warning assessments at risk": data.loc[data["early_warning"], "at_risk"].sum() == 408,
    "63 overdue assessments, all at risk":
        (~data["early_warning"]).sum() == 63 and data.loc[~data["early_warning"], "at_risk"].all(),
    "One row per request": requests["request_id"].is_unique,
    "Every assessment matches one request": data["request_id"].isin(requests["request_id"]).all(),
    "Assessment and request employee IDs agree":
        (assessments["employee_id"] == assessments["request_id"].map(requests.set_index("request_id")["employee_id"])).all(),
    "One employee type per employee": requests.groupby("employee_id")["employee_type"].nunique().eq(1).all(),
}, name="passed")
assert setup_checks.all(), "Data does not match the Exercise 02 and 04 documentation."
setup_checks.to_frame()

,passed
"2,579 assessments",True
"2,516 early-warning assessments",True
408 early-warning assessments at risk,True
"63 overdue assessments, all at risk",True
One row per request,True
Every assessment matches one request,True
Assessment and request employee IDs agree,True
One employee type per employee,True


## B. Fixed employee-level split

All of one employee's tasks and assessments are kept in the same split, so no employee contributes to both rule-setting and evaluation. Employees are divided approximately 60/20/20 into training, validation, and test, with seed 2026.

The split is stratified by employee type and the number of the employee's tasks that missed the deadline (0, 1, or 2+). Late tasks are counted once per task from `requests.csv`, not once per assessment. This uses outcome information only to improve balance between the splits; it is never a rule or model input. Random sampling can produce different at-risk rates between splits, and stratifying reduces that difference without guaranteeing identical groups.

Rounding within each stratum gives 299, 103, and 98 employees rather than exactly 300, 100, and 100. Some strata are small (for example, 7 permanent employees with two or more late tasks). The split is saved to `outputs/employee_split.csv` for Exercises 06–10. On later runs, the notebook rebuilds the split and stops if it differs from the saved file, so a change in the source data cannot silently change the evaluation groups.

- **Training:** choose rule thresholds (and, in Exercise 06, fit the model).
- **Validation:** compare baselines and models, and review errors, in Exercises 05–09.
- **Test:** untouched until one final evaluation after the model and threshold are fixed in Exercise 09.

In [3]:
# One row per employee; late tasks are counted once per task (an outcome, used only to balance the split)
employees = (requests.groupby("employee_id")
             .agg(employee_type=("employee_type", "first"), tasks=("request_id", "size"),
                  late_tasks=("ready_by_start", lambda s: (~s).sum()))
             .reset_index())
employees["late_group"] = pd.cut(employees["late_tasks"], [-1, 0, 1, np.inf], labels=["0", "1", "2+"]).astype(str)

rng = np.random.default_rng(SEED)
parts = []
for _, group in employees.groupby(["employee_type", "late_group"]):
    ids = group["employee_id"].to_numpy().copy()
    rng.shuffle(ids)
    position = np.arange(len(ids))
    labels = np.where(position < round(0.6 * len(ids)), "train",
                      np.where(position < round(0.8 * len(ids)), "validation", "test"))
    parts.append(pd.DataFrame({"employee_id": ids, "split": labels}))
employee_split = pd.concat(parts).sort_values("employee_id").reset_index(drop=True)

# Later exercises rely on this file, so a rerun must reproduce it exactly rather than silently replace it
split_file = OUTPUT_DIR / "employee_split.csv"
if split_file.exists():
    saved = pd.read_csv(split_file)
    assert saved.equals(employee_split), "The new split differs from the saved employee_split.csv; investigate before continuing."
    print("Split matches the saved employee_split.csv")
else:
    employee_split.to_csv(split_file, index=False)
    print("Saved a new employee_split.csv")

SPLITS = ["train", "validation", "test"]
strata = (employees.merge(employee_split, on="employee_id")
          .pivot_table(index=["employee_type", "late_group"], columns="split", values="employee_id",
                       aggfunc="size", fill_value=0)[SPLITS])
display(strata)

data = data.merge(employee_split, on="employee_id", how="left", validate="many_to_one")
split_checks = pd.Series({
    "Every employee assigned exactly once": employee_split["employee_id"].is_unique and len(employee_split) == len(employees),
    "Every assessment has a split": data["split"].notna().all(),
    "No task appears in more than one split": data.groupby("request_id")["split"].nunique().max() == 1,
}, name="passed")
assert split_checks.all(), "The split is inconsistent."
display(split_checks.to_frame())

early = data[data["early_warning"]]
split_summary = pd.DataFrame({
    "employees": employee_split["split"].value_counts(),
    "tasks": data.groupby("split")["request_id"].nunique(),
    "early_warning_assessments": early.groupby("split").size(),
    "early_warning_at_risk": early.groupby("split")["at_risk"].sum(),
    "overdue_assessments": data[~data["early_warning"]].groupby("split").size(),
}).reindex(SPLITS)
split_summary["at_risk_%"] = (split_summary["early_warning_at_risk"]
                              / split_summary["early_warning_assessments"] * 100).round(1)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv")
split_summary

Split matches the saved employee_split.csv


split                     train  validation  test
employee_type late_group                         
permanent     0             145          48    48
              1              10           3     3
              2+              4           2     1
seasonal      0              23           8     8
              1              14           5     5
              2+             40          14    13
temporary     0              34          12    11
              1              10           4     3
              2+             19           7     6

,passed
Every employee assigned exactly once,True
Every assessment has a split,True
No task appears in more than one split,True


,employees,tasks,early_warning_assessments,early_warning_at_risk,overdue_assessments,at_risk_%
split,,,,,,
train,299,1244,1509,246,30,16.3
validation,103,418,505,84,21,16.6
test,98,402,502,78,12,15.5


## C. Baseline rules and thresholds

Four baselines are compared:

- **Never at risk (majority baseline):** predicts that no task is at risk, matching the most common outcome.
- **Flag everything (reference strategy):** flags every assessment. It shows the maximum recall and the largest number of flags; it is not a useful operational baseline.
- **Days-until-start rule:** flags an assessment when `business_days_until_start` is at or below a threshold.
- **Days rule by employee type:** the same rule with a separate threshold for each employee type. Employee type is a job category, not a personal attribute.

**Threshold rule:** for each rule, choose the smallest days-until-start threshold that achieves at least 90% recall on training early-warning assessments. Because days are whole numbers, the chosen threshold can exceed 90% recall on training. An employee type uses its own threshold only if it has at least 20 at-risk training assessments; otherwise it uses the global threshold. The minimum of 20 is a practical choice, not a statistical guarantee, and thresholds based on few tasks or employees may be unstable, so the counts behind each threshold are shown. All thresholds, including the global one, are saved to `outputs/rule_thresholds.csv` so later exercises can reuse the exact rules.

A time-in-status rule is omitted to keep the comparison focused on simple rules supported by the exploratory findings in Exercise 04.

In [4]:
RECALL_TARGET = 0.90
MIN_POSITIVES = 20   # practical minimum of at-risk training assessments for a separate employee-type threshold
EMPLOYEE_TYPES = ["permanent", "temporary", "seasonal"]
train_early = early[early["split"] == "train"]

def smallest_threshold(group):
    """Smallest days-until-start threshold k where flagging days <= k reaches the recall target on this group."""
    at_risk_days = group.loc[group["at_risk"], "business_days_until_start"]
    for k in range(int(group["business_days_until_start"].min()), int(group["business_days_until_start"].max()) + 1):
        if (at_risk_days <= k).mean() >= RECALL_TARGET:
            return k
    raise ValueError("No threshold reaches the recall target.")

def support(group):
    """Training evidence behind a threshold: at-risk assessments, tasks, and employees."""
    at_risk = group[group["at_risk"]]
    return {"at_risk_assessments": len(at_risk), "at_risk_tasks": at_risk["request_id"].nunique(),
            "employees_with_at_risk": at_risk["employee_id"].nunique()}

global_threshold = smallest_threshold(train_early)
threshold_rows = [{"rule": "days_until_start_rule", "employee_type": "all", "threshold": global_threshold,
                   "source": "all training", **support(train_early)}]
for employee_type in EMPLOYEE_TYPES:
    group = train_early[train_early["employee_type"] == employee_type]
    enough = support(group)["at_risk_assessments"] >= MIN_POSITIVES
    threshold_rows.append({"rule": "days_rule_by_employee_type", "employee_type": employee_type,
                           "threshold": smallest_threshold(group) if enough else global_threshold,
                           "source": "own group" if enough else "global fallback", **support(group)})
rule_thresholds = pd.DataFrame(threshold_rows)
rule_thresholds.to_csv(OUTPUT_DIR / "rule_thresholds.csv", index=False)

type_thresholds = rule_thresholds.query("rule == 'days_rule_by_employee_type'").set_index("employee_type")["threshold"]
assert data["employee_type"].isin(type_thresholds.index).all(), "An employee type has no threshold."

BASELINES = {
    "never_at_risk": lambda d: pd.Series(False, index=d.index),
    "flag_everything": lambda d: pd.Series(True, index=d.index),
    "days_until_start_rule": lambda d: d["business_days_until_start"] <= global_threshold,
    "days_rule_by_employee_type": lambda d: d["business_days_until_start"] <= d["employee_type"].map(type_thresholds),
}
for name, rule in BASELINES.items():
    data[name] = rule(data)

print("Global threshold: flag when business days until start <=", global_threshold)
rule_thresholds

Global threshold: flag when business days until start <= 7


,rule,employee_type,threshold,source,at_risk_assessments,at_risk_tasks,employees_with_at_risk
0,days_until_start_rule,all,7,all training,246,202,92
1,days_rule_by_employee_type,permanent,9,own group,26,14,12
2,days_rule_by_employee_type,temporary,7,own group,75,60,28
3,days_rule_by_employee_type,seasonal,6,own group,145,128,52


## D. Baseline comparison

Each baseline is evaluated on training and validation early-warning assessments; test assessments are not evaluated. Recall (the share of at-risk assessments flagged) is the main measure because of the 90% target. Precision shows the share of flags that are at risk.

**Flagged assessments** is the number of assessments suggested for coordinator review. One task can be assessed and flagged at both checkpoints, so **unique tasks flagged** is also reported. Neither measures staff time. Accuracy is shown as a secondary measure: because most assessments are not at risk, a baseline can have high accuracy while flagging nothing.

The validation set is small and clustered by task and employee, which limits the precision of these results; small differences between baselines should not be treated as meaningful. The 21 overdue validation assessments are reported separately, and predictions for every validation assessment are saved.

In [5]:
def evaluate(group, flag):
    """Counts and measures for one baseline's flags on one group of assessments."""
    y, p = group["at_risk"], group[flag]
    tp, fp = (p & y).sum(), (p & ~y).sum()
    fn, tn = (~p & y).sum(), (~p & ~y).sum()
    return pd.Series({
        "assessments": len(group), "at_risk": y.sum(), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "recall": tp / (tp + fn) if tp + fn else np.nan, "precision": tp / (tp + fp) if tp + fp else np.nan,
        "flagged_assessments": p.sum(), "flagged_%": p.mean() * 100,
        "unique_tasks_flagged": group.loc[p, "request_id"].nunique(),
        "unique_tasks": group["request_id"].nunique(),
        "accuracy": (p == y).mean(),
    })

rows = []
for split in ["train", "validation"]:
    group = data[data["early_warning"] & (data["split"] == split)]
    for name in BASELINES:
        rows.append(evaluate(group, name).rename((split, name)))
comparison = pd.DataFrame(rows)
comparison.index.names = ["split", "baseline"]
comparison = comparison.astype({c: int for c in ["assessments", "at_risk", "TP", "FP", "FN", "TN",
                                                  "flagged_assessments", "unique_tasks_flagged", "unique_tasks"]})
comparison.round(3).to_csv(OUTPUT_DIR / "baseline_comparison.csv")

# Overdue validation assessments are reported separately: they are already past the deadline
overdue_validation = data[~data["early_warning"] & (data["split"] == "validation")]
overdue_flags = pd.Series({name: int(overdue_validation[name].sum()) for name in BASELINES},
                          name=f"flagged (of {len(overdue_validation)} overdue)")

# Saved predictions: every validation assessment, with each baseline's flag
prediction_columns = ["assessment_id", "request_id", "employee_id", "checkpoint", "employee_type",
                      "business_days_until_start", "early_warning", "at_risk"] + list(BASELINES)
validation_predictions = data.loc[data["split"] == "validation", prediction_columns]
validation_predictions.to_csv(OUTPUT_DIR / "baseline_predictions_validation.csv", index=False)
print(f"Saved predictions for {len(validation_predictions)} validation assessments")

display(overdue_flags.to_frame())
comparison[["at_risk", "TP", "FN", "recall", "precision", "flagged_assessments", "flagged_%",
            "unique_tasks_flagged", "unique_tasks", "accuracy"]].round(3)

Saved predictions for 526 validation assessments


,flagged (of 21 overdue)
never_at_risk,0
flag_everything,21
days_until_start_rule,21
days_rule_by_employee_type,21


at_risk   TP   FN  recall  precision  \
split      baseline                                                           
train      never_at_risk                   246    0  246   0.000        NaN   
           flag_everything                 246  246    0   1.000      0.163   
           days_until_start_rule           246  227   19   0.923      0.361   
           days_rule_by_employee_type      246  228   18   0.927      0.358   
validation never_at_risk                    84    0   84   0.000        NaN   
           flag_everything                  84   84    0   1.000      0.166   
           days_until_start_rule            84   80    4   0.952      0.349   
           days_rule_by_employee_type       84   79    5   0.940      0.345   

                                       flagged_assessments  flagged_%  \
split      baseline                                                     
train      never_at_risk                                 0      0.000   
           flag_everything                            1509    100.000   
           days_until_start_rule                       628     41.617   
           days_rule_by_employee_type                  637     42.213   
validation never_at_risk                                 0      0.000   
           flag_everything                             505    100.000   
           days_until_start_rule                       229     45.347   
           days_rule_by_employee_type                  229     45.347   

                                       unique_tasks_flagged  unique_tasks  \
split      baseline                                                         
train      never_at_risk                                  0          1214   
           flag_everything                             1214          1214   
           days_until_start_rule                        498          1214   
           days_rule_by_employee_type                   509          1214   
validation never_at_risk                                  0           397   
           flag_everything                              397           397   
           days_until_start_rule                        171           397   
           days_rule_by_employee_type                   175           397   

                                       accuracy  
split      baseline                              
train      never_at_risk                  0.837  
           flag_everything                0.163  
           days_until_start_rule          0.722  
           days_rule_by_employee_type     0.717  
validation never_at_risk                  0.834  
           flag_everything                0.166  
           days_until_start_rule          0.697  
           days_rule_by_employee_type     0.693

## E. Why this is a fair starting point

**Transparent and available at assessment time.** The rules use only `business_days_until_start` and `employee_type`. Both are documented model inputs available at assessment time. Exercise 04 checked the timing calculation and reviewed the selected input roles. Each rule can be read and applied by a person without a model.

**Separated from evaluation.** Every threshold was chosen on training employees only and evaluated on validation employees who were not used to choose it. Test data has not been used to choose thresholds or evaluate baseline performance.

**Measured the way the model will be.** The baselines use the same fixed split, the same early-warning group, and the same 90% recall target that Exercise 06's model will be compared against. This keeps the evaluation setup consistent, making later comparisons easier to interpret.

**Strong rather than easy to beat.** Exercise 04 showed that days until start is strongly related to the simulated outcome. A model that only beat the majority baseline would show little; comparing it with the days rule tests whether it adds anything beyond the simplest useful rule. The majority baseline shows why accuracy is misleading here, and flag-everything shows the largest possible number of flags.

**Results on validation.** Both rules met the 90% recall target: the global days rule (flag when 7 or fewer business days remain) reached 95.2% recall and the employee-type rule 94.0%. Both flagged 229 of 505 early-warning assessments (45%), and about one in three flags was at risk. The employee-type rule showed no improvement on this validation split: the global rule caught one more at-risk assessment with the same assessment flag count and fewer unique tasks flagged (171 compared with 175). Never at risk reached 83% accuracy while flagging no at-risk assessments.

**What this enables.** Exercise 06 will examine whether a model maintains at least 90% validation recall while reducing flags relative to the global rule, comparing flagged assessments and unique tasks flagged separately. The validation counts above are reference points for this split, not targets to carry unchanged into the differently sized test set.

**Limits.** These results describe rules applied to simulated outcomes, not real employees or HR performance. The validation set is small and clustered by task and employee, so small differences between baselines should not be treated as meaningful, and the permanent-employee threshold rests on 26 at-risk assessments from 12 employees.